# Esquema Estrella - Data Warehouse

Consolida datos históricos de reservaciones, productos, pedidos y usuarios en un esquema estrella dentro de Hive.

| Tabla | Tipo | Descripción |
|---|---|---|
| `dim_tiempo` | Dimensión | Fechas y horas desglosadas |
| `dim_usuario` | Dimensión | Datos de usuarios |
| `dim_plato` | Dimensión | Platos y categorías |
| `dim_mesa` | Dimensión | Mesas y restaurante |
| `dim_restaurante` | Dimensión | Datos del restaurante |
| `fact_pedidos` | Hechos | Un registro por ítem de pedido |
| `fact_reservaciones` | Hechos | Un registro por reservación |

## 0. Configuración

In [1]:
# ── Cambiá esta variable según la fuente activa ──────────────────
FUENTE = "mongo"   # "mongo" | "postgres"
# ─────────────────────────────────────────────────────────────────

HIVE_DB = "estrella"

# MongoDB
MONGO_URI = "mongodb://host.docker.internal:27017"
MONGO_DB  = "restaurantes_db"

# PostgreSQL
PG_HOST     = "host.docker.internal"
PG_PORT     = 4001
PG_DB       = "restaurantes_db"
PG_USER     = "postgres"
PG_PASSWORD = "postgres"
JDBC_URL    = f"jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DB}"
JDBC_PROPS  = {"user": PG_USER, "password": PG_PASSWORD, "driver": "org.postgresql.Driver"}

assert FUENTE in ("mongo", "postgres"), "FUENTE debe ser 'mongo' o 'postgres'"
print(f"Fuente seleccionada: {FUENTE.upper()}")

Fuente seleccionada: MONGO


## 1. Sesión de Spark

In [2]:
from pyspark.sql import SparkSession

builder = (
    SparkSession.builder
    .appName(f"Esquema Estrella — {FUENTE.upper()}")
    .master("spark://spark-master:7077")
    .config("spark.executor.memory", "1g")
    .config("spark.executor.cores", "1")
    .config("spark.sql.warehouse.dir", "/opt/hive/data/warehouse")
    .config("spark.sql.catalogImplementation", "hive")
)

if FUENTE == "mongo":
    builder = builder \
        .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.13:10.4.0") \
        .config("spark.mongodb.read.connection.uri", f"{MONGO_URI}/{MONGO_DB}")

spark = builder.enableHiveSupport().getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} listo.")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.mongodb.spark#mongo-spark-connector_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1f6e17a3-6e06-46af-a8fc-ca1607492c78;1.0
	confs: [default]
	found org.mongodb.spark#mongo-spark-connector_2.13;10.4.0 in central
	found org.mongodb#mongodb-driver-sync;5.1.4 in central
	[5.1.4] org.mongodb#mongodb-driver-sync;[5.1.1,5.1.99)
	found org.mongodb#bson;5.1.4 in central
	found org.mongodb#mongodb-driver-core;5.1.4 in central
	found org.mongodb#bson-record-codec;5.1.4 in central
:: resolution report :: resolve 3886ms :: artifacts dl 43ms
	:: modules in use:
	org.mongodb#bson;5.1.4 from central in [default]
	org.mongodb#bson-record-codec;5.1.4 from central in [default]
	org.mongodb#mongodb-driver-core;5.1.4 from central in [defaul

Spark 4.1.2 listo.


## 2. Lectura de datos

In [3]:
from pyspark.sql.functions import col, monotonically_increasing_id

if FUENTE == "mongo":
    def read_col(c):
        return spark.read.format("mongodb").option("collection", c).load()

    pedidos       = read_col("pedidos")
    platos        = read_col("platos")
    menus         = read_col("menus")
    reservaciones = read_col("reservaciones")
    usuarios      = read_col("usuarios")
    mesas         = read_col("mesas")
    restaurantes  = read_col("restaurantes")

else:
    def read_table(t):
        return spark.read.jdbc(url=JDBC_URL, table=t, properties=JDBC_PROPS)

    pedidos       = read_table("pedidos")
    item_pedido   = read_table("item_pedido")
    platos        = read_table("platos")
    menus         = read_table("menus")
    reservaciones = read_table("reservaciones")
    usuarios      = read_table("usuarios")
    mesas         = read_table("mesas")
    restaurantes  = read_table("restaurantes")

print("Datos cargados")

Datos cargados


## 3. Base de datos en Hive

In [4]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS {HIVE_DB}")
spark.sql(f"USE {HIVE_DB}")
print(f"Base de datos '{HIVE_DB}' lista.")

26/06/17 21:08:58 WARN HiveConf: HiveConf of name hive.metastore.local does not exist
26/06/17 21:08:58 WARN HiveClientImpl: Detected HiveConf hive.execution.engine is 'spark' and will be reset to 'mr' to disable useless hive logic


Base de datos 'estrella' lista.


---
## Dimensiones

In [5]:
from pyspark.sql.functions import (
    col, year, month, dayofmonth, hour, date_format,
    to_timestamp, to_date, coalesce, lit, explode,
    monotonically_increasing_id, regexp_extract, substring
)

def save(df, table):
    """Materializa y guarda en Hive."""
    data = df.collect()
    out  = spark.createDataFrame(data, df.schema)
    out.write.mode("overwrite").saveAsTable(f"{HIVE_DB}.{table}")
    print(f"{table}: {len(data):,} filas")
    return out

### dim_restaurante

In [6]:
if FUENTE == "mongo":
    dim_restaurante = restaurantes.select(
        col("_id").alias("restaurante_sk"),
        col("nombre_rest"),
        col("ubicacion"),
        col("correo_rest"),
        col("telefono_rest")
    )
else:
    dim_restaurante = restaurantes.select(
        col("id").cast("string").alias("restaurante_sk"),
        col("nombre_rest"),
        col("ubicacion"),
        col("correo_rest"),
        col("telefono_rest")
    )

save(dim_restaurante, "dim_restaurante")
spark.sql(f"SELECT * FROM {HIVE_DB}.dim_restaurante LIMIT 3").show(truncate=False)

26/06/17 21:09:28 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


dim_restaurante: 50 filas
+------------------------+-----------+--------------+----------------------+-------------+
|restaurante_sk          |nombre_rest|ubicacion     |correo_rest           |telefono_rest|
+------------------------+-----------+--------------+----------------------+-------------+
|6a2eda3e3e7b5b96675e6710|La Choza   |Tibás         |cecilia66@example.com |2778-1045    |
|6a2eda3e3e7b5b96675e6711|El Bohío   |Guanacaste    |timoteo58@example.org |2197-6621    |
|6a2eda3e3e7b5b96675e6712|La Tinaja  |Ciudad Quesada|cristina89@example.net|2133-2745    |
+------------------------+-----------+--------------+----------------------+-------------+



### dim_usuario

In [7]:
if FUENTE == "mongo":
    dim_usuario = usuarios.select(
        col("_id").alias("usuario_sk"),
        col("id").alias("usuario_id"),
        col("nombre"),
        col("email")
    )
else:
    dim_usuario = usuarios.select(
        col("id").alias("usuario_sk"),
        col("usuario"),
        col("nombre"),
        col("apellido"),
        col("email")
    )

save(dim_usuario, "dim_usuario")
spark.sql(f"SELECT * FROM {HIVE_DB}.dim_usuario LIMIT 3").show(truncate=False)

dim_usuario: 1 filas
+------------------------+------------------------------------+---------+-------------+
|usuario_sk              |usuario_id                          |nombre   |email        |
+------------------------+------------------------------------+---------+-------------+
|6a2712f14e5d81e4a322e95a|61984740-a09a-40e7-855e-2942e1046413|evan test|evan@test.com|
+------------------------+------------------------------------+---------+-------------+



### dim_plato

In [8]:
if FUENTE == "mongo":
    dim_plato = platos.join(
        menus.select(col("_id").alias("menu_id_join"), col("nombre_menu").alias("categoria")),
        platos.menu_id == col("menu_id_join"), how="left"
    ).select(
        col("_id").alias("plato_sk"),
        col("nombre_plato"),
        col("precio"),
        col("categoria"),
        col("descripcion")
    )
else:
    dim_plato = platos.join(
        menus.select(col("id").alias("menu_id_join"), col("nombre_menu").alias("categoria")),
        platos.menu_id == col("menu_id_join"), how="left"
    ).select(
        col("id").cast("string").alias("plato_sk"),
        col("nombre_plato"),
        col("precio"),
        col("categoria"),
        col("descripcion")
    )

save(dim_plato, "dim_plato")
spark.sql(f"SELECT * FROM {HIVE_DB}.dim_plato LIMIT 3").show(truncate=False)

dim_plato: 4,879 filas
+------------------------+----------------+------+---------+----------------------------------------------------------------------+
|plato_sk                |nombre_plato    |precio|categoria|descripcion                                                           |
+------------------------+----------------+------+---------+----------------------------------------------------------------------+
|6a2eda3f3e7b5b96675e76f2|Sopa minestrone |4176  |Sopas    |Pregunta siendo constitución esa quería hablar españoles modelo tiene.|
|6a2eda3f3e7b5b96675e76f3|Caldo tlalpeño  |3903  |Sopas    |Grande económica dicho servicios ellos mi poco sigue supuesto tenemos.|
|6a2eda3f3e7b5b96675e76f4|Sopa de lentejas|3292  |Sopas    |El todas dinero pasado quiere norte tener deseo.                      |
+------------------------+----------------+------+---------+----------------------------------------------------------------------+



### dim_mesa

In [9]:
if FUENTE == "mongo":
    dim_mesa = mesas.join(
        restaurantes.select(col("_id").alias("rest_id_join"), col("nombre_rest")),
        mesas.rest_id == col("rest_id_join"), how="left"
    ).select(
        col("_id").alias("mesa_sk"),
        col("numero_mesa"),
        col("capacidad"),
        col("disponible"),
        col("nombre_rest").alias("restaurante")
    )
else:
    dim_mesa = mesas.join(
        restaurantes.select(col("id").alias("rest_id_join"), col("nombre_rest")),
        mesas.rest_id == col("rest_id_join"), how="left"
    ).select(
        col("id").cast("string").alias("mesa_sk"),
        col("numero_mesa"),
        col("capacidad"),
        col("disponible"),
        col("nombre_rest").alias("restaurante")
    )

save(dim_mesa, "dim_mesa")
spark.sql(f"SELECT * FROM {HIVE_DB}.dim_mesa LIMIT 3").show(truncate=False)

dim_mesa: 451 filas
+------------------------+-----------+---------+----------+-----------+
|mesa_sk                 |numero_mesa|capacidad|disponible|restaurante|
+------------------------+-----------+---------+----------+-----------+
|6a2eda3f3e7b5b96675e7d09|1          |2        |true      |La Tinaja  |
|6a2eda3f3e7b5b96675e7d0a|2          |4        |true      |La Tinaja  |
|6a2eda3f3e7b5b96675e7d0b|3          |4        |true      |La Tinaja  |
+------------------------+-----------+---------+----------+-----------+



### dim_tiempo

In [10]:
# Recopilar todas las fechas únicas de pedidos y reservaciones
if FUENTE == "mongo":
    fechas_pedidos = pedidos.select(
        col("fecha_orden").alias("fecha_str")
    )
    fechas_reservas = reservaciones.select(
        col("dia_reservacion").alias("fecha_str")
    )
    fechas = fechas_pedidos.union(fechas_reservas).distinct()
    dim_tiempo = fechas.withColumn("fecha", to_date(col("fecha_str"), "yyyy-MM-dd")) \
        .filter(col("fecha").isNotNull()) \
        .select(
            date_format(col("fecha"), "yyyyMMdd").alias("tiempo_sk"),
            col("fecha"),
            year(col("fecha")).alias("anio"),
            month(col("fecha")).alias("mes"),
            date_format(col("fecha"), "MMMM").alias("nombre_mes"),
            dayofmonth(col("fecha")).alias("dia"),
            date_format(col("fecha"), "EEEE").alias("dia_semana")
        ).distinct()
else:
    fechas_pedidos = pedidos.select(col("fecha_orden").cast("string").alias("fecha_str"))
    fechas_reservas = reservaciones.select(col("dia_reservacion").cast("string").alias("fecha_str"))
    fechas = fechas_pedidos.union(fechas_reservas).distinct()
    dim_tiempo = fechas.withColumn("fecha", to_date(col("fecha_str"), "yyyy-MM-dd")) \
        .filter(col("fecha").isNotNull()) \
        .select(
            date_format(col("fecha"), "yyyyMMdd").alias("tiempo_sk"),
            col("fecha"),
            year(col("fecha")).alias("anio"),
            month(col("fecha")).alias("mes"),
            date_format(col("fecha"), "MMMM").alias("nombre_mes"),
            dayofmonth(col("fecha")).alias("dia"),
            date_format(col("fecha"), "EEEE").alias("dia_semana")
        ).distinct()

save(dim_tiempo, "dim_tiempo")
spark.sql(f"SELECT * FROM {HIVE_DB}.dim_tiempo ORDER BY fecha LIMIT 5").show(truncate=False)

dim_tiempo: 313 filas
+---------+----------+----+---+----------+---+----------+
|tiempo_sk|fecha     |anio|mes|nombre_mes|dia|dia_semana|
+---------+----------+----+---+----------+---+----------+
|20250614 |2025-06-14|2025|6  |June      |14 |Saturday  |
|20250615 |2025-06-15|2025|6  |June      |15 |Sunday    |
|20250616 |2025-06-16|2025|6  |June      |16 |Monday    |
|20250617 |2025-06-17|2025|6  |June      |17 |Tuesday   |
|20250618 |2025-06-18|2025|6  |June      |18 |Wednesday |
+---------+----------+----+---+----------+---+----------+



---
## Tablas de hechos

### fact_pedidos

In [11]:
if FUENTE == "mongo":
    # Explotar items del array dentro del pedido
    items = pedidos.select(
        col("_id").alias("pedido_id"),
        col("usuario_id"),
        col("fecha_orden"),
        col("estado"),
        col("tipo_pedido"),
        explode(col("items")).alias("item")
    ).select(
        col("pedido_id"),
        col("usuario_id").alias("usuario_sk"),
        col("item.plato_id").alias("plato_sk"),
        col("item.cantidad").alias("cantidad"),
        date_format(to_date(col("fecha_orden"), "yyyy-MM-dd"), "yyyyMMdd").alias("tiempo_sk"),
        col("estado"),
        col("tipo_pedido")
    )
    # Enriquecer con precio del plato
    fact_pedidos = items.join(
        platos.select(col("_id").alias("plato_id_join"), col("precio")),
        items.plato_sk == col("plato_id_join"), how="left"
    ).select(
        col("pedido_id"),
        col("usuario_sk"),
        col("plato_sk"),
        col("tiempo_sk"),
        col("cantidad"),
        (col("cantidad") * col("precio")).alias("precio_total"),
        col("estado"),
        col("tipo_pedido")
    )
else:
    items = item_pedido.join(
        pedidos.select(
            col("id").alias("pedido_id"),
            col("usuario_id"),
            col("fecha_orden"),
            col("estado"),
            col("tipo_pedido")
        ),
        item_pedido.pedido_id == col("pedido_id"), how="left"
    )
    fact_pedidos = items.join(
        platos.select(col("id").alias("plato_id_join"), col("precio")),
        items.plato_id == col("plato_id_join"), how="left"
    ).select(
        col("pedido_id").cast("string"),
        col("usuario_id").alias("usuario_sk"),
        col("plato_id").cast("string").alias("plato_sk"),
        date_format(col("fecha_orden"), "yyyyMMdd").alias("tiempo_sk"),
        col("cantidad"),
        (col("cantidad") * col("precio")).alias("precio_total"),
        col("estado"),
        col("tipo_pedido")
    )

save(fact_pedidos, "fact_pedidos")
spark.sql(f"SELECT * FROM {HIVE_DB}.fact_pedidos LIMIT 5").show(truncate=False)

fact_pedidos: 1,236 filas
+------------------------+------------------------------------+------------------------+---------+--------+------------+--------------+-----------+
|pedido_id               |usuario_sk                          |plato_sk                |tiempo_sk|cantidad|precio_total|estado        |tipo_pedido|
+------------------------+------------------------------------+------------------------+---------+--------+------------+--------------+-----------+
|6a2eda3f3e7b5b96675e7eaa|7f2cfe46-068a-468b-9ec7-b54ca9c8e240|6a2eda3f3e7b5b96675e7aa3|20250703 |2       |9974        |listo         |delivery   |
|6a2eda3f3e7b5b96675e7ed0|b436c285-d870-4722-ab4e-36fa43e4092e|6a2eda3f3e7b5b96675e71b3|20251103 |2       |12492       |cancelado     |para llevar|
|6a2eda3f3e7b5b96675e7eda|c9a54d1e-361e-47c0-8fd6-c197108f7e55|6a2eda3f3e7b5b96675e6bb9|20251128 |1       |6166        |entregado     |para llevar|
|6a2eda3f3e7b5b96675e7ef9|274a51ed-8606-4b1d-a550-9e26530826f3|6a2eda3f3e7b5b96675e762

### fact_reservaciones

In [12]:
if FUENTE == "mongo":
    fact_reservaciones = reservaciones.select(
        col("_id").alias("reservacion_sk"),
        col("usuario_id").alias("usuario_sk"),
        col("mesa_id").alias("mesa_sk"),
        date_format(to_date(col("dia_reservacion"), "yyyy-MM-dd"), "yyyyMMdd").alias("tiempo_sk"),
        col("hora_reservacion"),
        col("estado")
    )
else:
    fact_reservaciones = reservaciones.select(
        col("id").cast("string").alias("reservacion_sk"),
        col("usuario_id").alias("usuario_sk"),
        col("mesa_id").cast("string").alias("mesa_sk"),
        date_format(col("dia_reservacion"), "yyyyMMdd").alias("tiempo_sk"),
        col("hora_reservacion").cast("string"),
        col("estado")
    )

save(fact_reservaciones, "fact_reservaciones")
spark.sql(f"SELECT * FROM {HIVE_DB}.fact_reservaciones LIMIT 5").show(truncate=False)

fact_reservaciones: 200 filas
+------------------------+------------------------------------+------------------------+---------+----------------+----------+
|reservacion_sk          |usuario_sk                          |mesa_sk                 |tiempo_sk|hora_reservacion|estado    |
+------------------------+------------------------------------+------------------------+---------+----------------+----------+
|6a2eda3f3e7b5b96675e7ddc|237f3eab-78c4-40c0-a5d1-0ae39d4b4021|6a2eda3f3e7b5b96675e7d70|20250803 |18:30           |cancelada |
|6a2eda3f3e7b5b96675e7ddd|eea1321b-8e9c-4377-9436-9c7b71404ffd|6a2eda3f3e7b5b96675e7c33|20251108 |17:00           |cancelada |
|6a2eda3f3e7b5b96675e7dde|85ad5b6f-ae24-48ee-b8c2-1d9d7c76ede6|6a2eda3f3e7b5b96675e7c77|20260105 |17:00           |completada|
|6a2eda3f3e7b5b96675e7ddf|237f3eab-78c4-40c0-a5d1-0ae39d4b4021|6a2eda3f3e7b5b96675e7c9e|20260524 |14:30           |completada|
|6a2eda3f3e7b5b96675e7de0|274a51ed-8606-4b1d-a550-9e26530826f3|6a2eda3f3e7b5b9667

---
## Resumen del esquema

In [13]:
print(f"Tablas en '{HIVE_DB}':")
spark.sql(f"SHOW TABLES IN {HIVE_DB}").show(truncate=False)

print("\nConteo de filas:")
for t in ["dim_restaurante", "dim_usuario", "dim_plato", "dim_mesa", "dim_tiempo", "fact_pedidos", "fact_reservaciones"]:
    n = spark.sql(f"SELECT COUNT(*) AS n FROM {HIVE_DB}.{t}").first()["n"]
    print(f"  {t:<25} {n:>6,} filas")

Tablas en 'estrella':
+---------+------------------+-----------+
|namespace|tableName         |isTemporary|
+---------+------------------+-----------+
|estrella |dim_mesa          |false      |
|estrella |dim_plato         |false      |
|estrella |dim_restaurante   |false      |
|estrella |dim_tiempo        |false      |
|estrella |dim_usuario       |false      |
|estrella |fact_pedidos      |false      |
|estrella |fact_reservaciones|false      |
+---------+------------------+-----------+


Conteo de filas:
  dim_restaurante               50 filas
  dim_usuario                    1 filas
  dim_plato                  4,879 filas
  dim_mesa                     451 filas
  dim_tiempo                   313 filas
  fact_pedidos               1,236 filas
  fact_reservaciones           200 filas


---
## Cerrar Sesión

In [14]:
spark.stop()
print("\n Sesión cerrada.")


 Sesión cerrada.


26/06/17 21:29:25 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/spark-1ca75ffe-a078-4ed1-b4e8-925ab5fd86b6. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/spark-1ca75ffe-a078-4ed1-b4e8-925ab5fd86b6
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:352)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:269)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:248)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:158)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:157)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1062)
	at org.apache.spark.util.ShutdownHookManager$.$anonfun$new$4(ShutdownHookManager.scala:70)
	at org.apache.spark.util.ShutdownHookManager$.$anonfun$new$4$adapted(ShutdownHookManager.scala:67)
	at scala.collection.ArrayOps$.foreach$exten